In [ ]:
pip install numpy scikit-learn

In [ ]:

import re
import math
import numpy as np

from collections import Counter
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# -----------------------------------------
# 1. Training dataset
# Each inner list represents one document.
# -----------------------------------------

training_documents = [
    [
        "Artificial intelligence enables machines to perform tasks.",
        "Machine learning is an important branch of artificial intelligence.",
        "Deep learning uses neural networks."
    ],
    [
        "Natural language processing analyzes human language.",
        "Text classification is an NLP application.",
        "Tokenization divides text into smaller units."
    ],
    [
        "Cricket is a popular sport in India.",
        "Players score runs by hitting the ball.",
        "The team with more runs wins."
    ],
    [
        "Python is a popular programming language.",
        "It supports object oriented programming.",
        "Python is widely used in software development."
    ],
    [
        "Climate change affects global temperatures.",
        "Carbon emissions contribute to global warming.",
        "Renewable energy can reduce pollution."
    ],
    [
        "The solar system contains eight planets.",
        "Earth revolves around the Sun.",
        "Jupiter is the largest planet."
    ],
    [
        "Database systems store structured information.",
        "SQL is used to query relational databases.",
        "Database indexing improves search performance."
    ],
    [
        "Football is played by two competing teams.",
        "Players attempt to score goals.",
        "The goalkeeper protects the goal."
    ]
]

# -----------------------------------------
# 2. Sentence feature extraction
# -----------------------------------------

def extract_features(previous, current, first=False):

    prev_words = set(re.findall(r"\b\w+\b", previous.lower()))
    curr_words = set(re.findall(r"\b\w+\b", current.lower()))

    intersection = prev_words & curr_words
    union = prev_words | curr_words

    similarity = (
        len(intersection) / len(union)
        if union else 0
    )

    features = {
        "similarity": similarity,
        "first_sentence": int(first),
        "same_topic_overlap": len(intersection)
    }

    for word in curr_words:
        features["current_" + word] = 1

    for word in prev_words:
        features["previous_" + word] = 1

    return features


# -----------------------------------------
# 3. Prepare training samples
# B = Beginning of a new document
# I = Inside an existing document
# -----------------------------------------

X_train = []
y_train = []
training_sequences = []

for document in training_documents:

    sequence = []
    previous = ""

    for index, sentence in enumerate(document):

        label = "B" if index == 0 else "I"

        features = extract_features(
            previous,
            sentence,
            first=(index == 0)
        )

        X_train.append(features)
        y_train.append(label)
        sequence.append(label)

        previous = sentence

    training_sequences.append(sequence)


# -----------------------------------------
# 4. Train probabilistic classifier
# -----------------------------------------

vectorizer = DictVectorizer(sparse=True)

X_vectorized = vectorizer.fit_transform(X_train)

classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

classifier.fit(X_vectorized, y_train)

print("Probabilistic classifier trained.")


# -----------------------------------------
# 5. Learn finite-state transition model
# -----------------------------------------

states = ["B", "I"]

transition_counts = {
    "B": Counter(),
    "I": Counter()
}

initial_counts = Counter()

for sequence in training_sequences:

    initial_counts[sequence[0]] += 1

    for previous, current in zip(
        sequence[:-1], sequence[1:]
    ):
        transition_counts[previous][current] += 1

# Laplace smoothing

transition_prob = {}

for previous in states:

    total = sum(transition_counts[previous].values()) + 2

    for current in states:

        transition_prob[(previous, current)] = (
            transition_counts[previous][current] + 1
        ) / total

initial_prob = {
    state: (initial_counts[state] + 1) /
           (sum(initial_counts.values()) + 2)
    for state in states
}


# -----------------------------------------
# 6. Viterbi finite-state sequence model
# -----------------------------------------

def viterbi_decode(probabilities):

    n = len(probabilities)

    if n == 0:
        return []

    dp = [{} for _ in range(n)]
    backpointer = [{} for _ in range(n)]

    # Initial state probabilities
    for state in states:

        emission = max(probabilities[0][state], 1e-12)

        dp[0][state] = (
            math.log(initial_prob[state])
            + math.log(emission)
        )

    # Dynamic programming
    for t in range(1, n):

        for current in states:

            emission = max(
                probabilities[t][current], 1e-12
            )

            candidates = {
                previous:
                    dp[t-1][previous]
                    + math.log(
                        transition_prob[(previous, current)]
                    )
                    + math.log(emission)
                for previous in states
            }

            best_previous = max(
                candidates, key=candidates.get
            )

            dp[t][current] = candidates[best_previous]
            backpointer[t][current] = best_previous

    # Find best final state
    last_state = max(dp[-1], key=dp[-1].get)

    # Backtracking
    result = [last_state]

    for t in range(n - 1, 0, -1):

        last_state = backpointer[t][last_state]
        result.append(last_state)

    return result[::-1]


# -----------------------------------------
# 7. Document separation
# -----------------------------------------

def separate_documents(text):

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text.strip()
    )

    sentences = [s.strip() for s in sentences if s.strip()]

    if not sentences:
        return []

    features = []
    previous = ""

    for index, sentence in enumerate(sentences):

        features.append(
            extract_features(
                previous,
                sentence,
                first=(index == 0)
            )
        )

        previous = sentence

    X = vectorizer.transform(features)

    predicted_probabilities = classifier.predict_proba(X)

    probabilities = []

    for row in predicted_probabilities:

        probabilities.append({
            state: row[list(classifier.classes_).index(state)]
            for state in states
        })

    # Apply finite-state sequence modeling
    labels = viterbi_decode(probabilities)

    # Map sentence sequence into documents
    documents = []
    current_document = []

    for sentence, label in zip(sentences, labels):

        if label == "B" and current_document:

            documents.append(" ".join(current_document))
            current_document = []

        current_document.append(sentence)

    if current_document:
        documents.append(" ".join(current_document))

    return documents, sentences, labels


# -----------------------------------------
# 8. Test input
# -----------------------------------------

test_text = """
Artificial intelligence helps machines learn.
Neural networks are used in deep learning.
Natural language processing analyzes text.
Tokenization is an important NLP technique.
Cricket is played by two teams.
Players score runs during a match.
Python is used for programming.
Software developers use Python libraries.
"""

documents, sentences, predicted_labels = separate_documents(
    test_text
)

print("\nSEPARATED DOCUMENTS")

for index, document in enumerate(documents, 1):

    print(f"\nDocument {index}:")
    print(document)


# -----------------------------------------
# 9. Evaluation
# -----------------------------------------

# Expected boundaries for this test example
actual_labels = [
    "B", "I",
    "B", "I",
    "B", "I",
    "B", "I"
]

print("\nEVALUATION RESULTS")

print(
    "Accuracy:",
    accuracy_score(actual_labels, predicted_labels)
)

print(
    "Precision:",
    precision_score(
        actual_labels,
        predicted_labels,
        pos_label="B",
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        actual_labels,
        predicted_labels,
        pos_label="B",
        zero_division=0
    )
)

print(
    "F1 Score:",
    f1_score(
        actual_labels,
        predicted_labels,
        pos_label="B",
        zero_division=0
    )
)

print("\nClassification Report:")

print(
    classification_report(
        actual_labels,
        predicted_labels,
        labels=["B", "I"],
        zero_division=0
    )
)

print("\nActual labels:   ", actual_labels)
print("Predicted labels:", predicted_labels)


Probabilistic classifier trained.

SEPARATED DOCUMENTS

Document 1:
Artificial intelligence helps machines learn. Neural networks are used in deep learning. Natural language processing analyzes text. Tokenization is an important NLP technique. Cricket is played by two teams. Players score runs during a match. Python is used for programming. Software developers use Python libraries.

EVALUATION RESULTS
Accuracy: 0.625
Precision: 1.0
Recall: 0.25
F1 Score: 0.4

Classification Report:
              precision    recall  f1-score   support

           B       1.00      0.25      0.40         4
           I       0.57      1.00      0.73         4

    accuracy                           0.62         8
   macro avg       0.79      0.62      0.56         8
weighted avg       0.79      0.62      0.56         8


Actual labels:    ['B', 'I', 'B', 'I', 'B', 'I', 'B', 'I']
Predicted labels: ['B', 'I', 'I', 'I', 'I', 'I', 'I', 'I']
